## 1. Configuración e imports

In [1]:
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path
from tqdm.auto import tqdm

BASE_DIR = Path().resolve().parent
sys.path.append(str(BASE_DIR / 'backend'))
from features import extract_features

FMA_SMALL_DIR = BASE_DIR / 'data' / 'fma_small'
METADATA_DIR  = BASE_DIR / 'data' / 'fma_metadata'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True, parents=True)

## 2. Carga de metadatos y géneros

In [2]:
tracks = pd.read_csv(METADATA_DIR / 'tracks.csv', index_col=0, header=[0,1])
small = tracks[tracks['set','subset'] == 'small']
genres = small['track', 'genre_top'].dropna()

print("Distribución de géneros:")
print(genres.value_counts())

Distribución de géneros:
(track, genre_top)
Hip-Hop          1000
Pop              1000
Folk             1000
Experimental     1000
Rock             1000
International    1000
Electronic       1000
Instrumental     1000
Name: count, dtype: int64


## 3. Extracción de features con barra de progreso

In [3]:
features_list = []
labels_list = []
fallidos = 0
procesados = 0

for track_id, genre in tqdm(genres.items(), total=len(genres), desc="Extrayendo features"):
    track_id_str = str(track_id).zfill(6)
    folder = track_id_str[:3]
    audio_path = FMA_SMALL_DIR / folder / f"{track_id_str}.mp3"
    
    try:
        features = extract_features(audio_path)
        features_list.append(features)
        labels_list.append(genre)
        procesados += 1
    except Exception as e:
        print(f"Error procesando track {track_id}: {e}")
        fallidos += 1

Extrayendo features:   0%|          | 0/8000 [00:00<?, ?it/s]

c:\Users\danie\OneDrive\Escritorio\Clasificacion musical\.venv\Lib\site-packages\librosa\core\pitch.py:103: UserWarning: Trying to estimate tuning from empty frequency set.
  return pitch_tuning(
C:\Users\danie\OneDrive\Escritorio\Clasificacion musical\musicclassifier\backend\features.py:26: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=22050, mono=True, duration=30.0)
c:\Users\danie\OneDrive\Escritorio\Clasificacion musical\.venv\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
c:\Users\danie\OneDrive\Escritorio\Clasificacion musical\.venv\Lib\site-packages\librosa\core\spectrum.py:266: UserWarning: n_fft=1024 is too large for input signal of length=554
  warnings.warn(
c:\Users\danie\OneDrive\Escritorio\Clasificacion musical\.venv\Lib\site-

Error procesando track 99134: Error procesando el archivo de audio con librosa: 
Error procesando track 108925: Error procesando el archivo de audio con librosa: 
Error procesando track 133297: Error procesando el archivo de audio con librosa: 


## 4. Guardado de resultados

In [4]:
X = pd.DataFrame(features_list)
y = pd.Series(labels_list, name='genre')

X.to_csv(PROCESSED_DIR / 'X_clean.csv', index=False)
y.to_csv(PROCESSED_DIR / 'y_clean.csv', index=False)

print(f"Shape de X: {X.shape}")
print(f"Distribución de clases en y:\n{y.value_counts()}")
print(f"NaNs en X: {X.isna().sum().sum()}")
print(f"✅ Extracción completada: {procesados} tracks procesados, {fallidos} fallidos")

Shape de X: (7997, 355)
Distribución de clases en y:
genre
Hip-Hop          1000
Pop              1000
Folk             1000
International    1000
Instrumental     1000
Experimental      999
Rock              999
Electronic        999
Name: count, dtype: int64
NaNs en X: 0
✅ Extracción completada: 7997 tracks procesados, 3 fallidos
